# Scaffold Experiment Walkthrough

This notebook runs a small synthetic Aegis experiment with an inline config and an explicit model registry. It is scaffold evidence only: it demonstrates mechanics and artifact shape, not validated trading methodology, empirical edge, or investment advice.

In [ ]:
# ruff: noqa: E402, I001
from __future__ import annotations

import shutil
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'research').exists():
            return path
    raise RuntimeError('Run this notebook from inside the aegis-rd repository')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.aegis_research.component_registry import discover_component_registry
from research.aegis_research.config import resolve_lane_config
from research.aegis_research.experiments import run_experiment
from research.aegis_research.indicators import build_component_indicator_result
from research.aegis_research.labels import (
    LabelConfig,
    LabelGeneratorConfig,
    LabelTargetConfig,
    LabelTargetSelectionConfig,
    LabelTargetTransformConfig,
    build_label_result,
)
from research.aegis_research.model_plugins import make_default_model_registry

def write_component_registry(root: Path) -> Path:
    component_root = root / 'research' / 'components'
    indicator_root = component_root / 'indicators'
    label_root = component_root / 'labels'
    indicator_root.mkdir(parents=True, exist_ok=True)
    label_root.mkdir(parents=True, exist_ok=True)
    (indicator_root / 'returns.py').write_text('''
# %% define component metadata
COMPONENT_MANIFEST = {'family': 'indicators', 'id': 'example.returns', 'version': '1.0.0', 'input_names': ['Close'], 'param_names': [], 'output_names': ['returns'], 'default_outputs': ['returns'], 'default_model_features': [{'output': 'returns', 'transform': 'identity'}], 'supported_transforms': ['identity']}
COMPONENT_CALLABLE = 'run'

# %% main compute
def run(data):
    """Compute simple returns over the run-provided Close feature."""

    return data.feature('Close').pct_change().fillna(0.0)
'''.lstrip())
    (label_root / 'fixlb.py').write_text('''
# %% define component metadata
COMPONENT_MANIFEST = {'family': 'labels', 'id': 'example.fixlb', 'version': '1.0.0', 'input_names': ['Close'], 'target_role': 'supervised_target', 'target_kind': 'binary_classification', 'output_names': ['labels'], 'split_safety': {'purging_required': True}}
COMPONENT_CALLABLE = 'run'

# %% main compute
def run(data):
    """Placeholder component; the notebook supplies labels explicitly."""

    raise RuntimeError('this notebook passes an explicit label_result_builder')
'''.lstrip())
    return component_root


## Inline Config

The config is embedded here instead of loaded from `research/configs/experiments/`. The synthetic data, fixed FIXLB target, uncalibrated probabilities, fixed thresholds, execution assumptions, portfolio sizing, and report gates are all scaffold choices for learning the pipeline.

In [ ]:
SCAFFOLD_CONFIG = {
    'schema_version': 4,
    'lane': 'train',
    'name': 'synthetic_scaffold_notebook',
    'data': {
        'source': 'synthetic',
        'symbols': ['SYN'],
        'start': '2020-01-01',
        'timeframe': '1D',
        'rows': 240,
        'seed': 42,
        'arrays': ['OHLCV'],
    },
    'indicators': [{'source': 'component', 'ids': ['example.returns']}],
    'labeler': {'id': 'example.fixlb'},
    'train': {
        'model': {
            'source': 'plugin',
            'id': 'aegis.sklearn_logistic',
            'min_train_samples': 50,
            'params': {'max_iter': 1000, 'random_state': 42},
        },
        'split': {
            'kind': 'purged_kfold',
            'n_folds': 3,
            'n_test_folds': 1,
            'purge_td': '0D',
            'embargo_td': '0D',
            'max_splits': 3,
            'max_estimated_output_cells': 500000,
            'max_public_artifact_bytes': 5000000,
        },
        'signals': {
            'policy': 'long_only_hysteresis',
            'long_entry_threshold': 0.55,
            'long_exit_threshold': 0.50,
            'execution_timing': 'next_open',
        },
    },
    'portfolio': {
        'init_cash': 10000.0,
        'fees': 0.001,
        'slippage': 0.0005,
        'entry_budget': 1.0,
        'direction': 'longonly',
    },
    'report': {
        'freq': '1D',
        'year_freq': '252D',
        'min_oos_sharpe': 0.5,
        'max_oos_drawdown': 0.35,
        'min_oos_trades': 1,
    },
}


## Run With An Explicit Registry

The registry is built in Python before config resolution. Do not put import paths, estimator definitions, API keys, provider tokens, or credentials in YAML or notebooks; use environment-backed secret references for real provider credentials.

In [ ]:
scratch = TemporaryDirectory(prefix='aegis-scaffold-')
scratch_root = Path(scratch.name)
output_dir = Path('runs') / scratch_root.name
component_registry = discover_component_registry(
    root=write_component_registry(scratch_root),
    repo_root=scratch_root,
)
registry = make_default_model_registry()
resolved = resolve_lane_config(
    {**SCAFFOLD_CONFIG, 'output_dir': output_dir.as_posix()},
    component_registry=component_registry,
    model_registry=registry,
    expected_lane='train',
)
label_config = LabelConfig(
    generator=LabelGeneratorConfig(params={'n': 5}),
    target=LabelTargetConfig(
        select=LabelTargetSelectionConfig(params={'n': 5}),
        transform=LabelTargetTransformConfig(params={'threshold': 0.0}),
    ),
)

def label_result_builder(data):
    return build_label_result(
        data.feature('Close'),
        label_config,
        high=data.feature('High'),
        low=data.feature('Low'),
    )

def indicator_result_builder(data):
    return build_component_indicator_result(
        data,
        resolved.config.indicators,
        component_registry=component_registry,
    )

result = run_experiment(
    resolved,
    label_result_builder=label_result_builder,
    indicator_result_builder=indicator_result_builder,
)
report = result['report']
report['status']


## Inspect Mechanics-Only Output

The status, gates, and metrics below show how the scaffold reports evidence. They are mechanics-only output from synthetic data and should not be read as a passing strategy, empirical edge, or investment advice.

In [ ]:
{
    'status': report['status'],
    'reasons': report['reasons'],
    'validation': report['validation'],
}


In [ ]:
scratch.cleanup()
shutil.rmtree(output_dir, ignore_errors=True)
